In [1]:
from amt_tools.models.common import TranscriptionModel, SoftmaxGroups
from amt_tools import tools

# Regular imports
import torch
from torch import nn

class ResBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1):
        super(ResBlock, self).__init__()
        padding = kernel_size // 2  # Preserve spatial dimensions
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size, stride=stride, padding=padding)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size, stride=1, padding=padding)
        self.shortcut = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride)
        self.batchnorm = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        
    def forward(self, x):
        shortcut = x
        out = self.conv1(x)
        out = self.batchnorm(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.batchnorm(out)
        # put shortcut through fully connected layer to match dimensions
        shortcut = self.shortcut(shortcut)
        out += shortcut

        return out


class TabCNN(TranscriptionModel):
    """
    Implements the TabCNN model (http://archives.ismir.net/ismir2019/paper/000033.pdf).
    """

    def __init__(self, dim_in, profile, in_channels=1, model_complexity=1, device='cpu'):
        """
        Initialize the model and establish parameter defaults in function signature.

        Parameters
        ----------
        See TranscriptionModel class for others...
        """

        super().__init__(dim_in, profile, in_channels, model_complexity, 9, device)

        # Initialize a flag to check whether to pad input features
        self.online = False

        # Number of filters for each stage
        nf1 = 32 * self.model_complexity
        nf2 = 64 * self.model_complexity
        nf3 = nf2

        # Kernel size for each stage
        ks1 = (3, 3)
        ks2 = ks1
        ks3 = ks1

        # Reduction size for each stage
        rd1 = (2, 2)

        # Dropout percentages for each stage
        dp1 = 0.25
        dp2 = 0.50

        self.conv = nn.Sequential(
            # 1st convolution
            nn.Conv2d(self.in_channels, nf1, ks1),
            # Activation function
            nn.ReLU(inplace=True),
            # 2nd convolution
            nn.Conv2d(nf1, nf2, ks2),
            # Activation function
            nn.ReLU(inplace=True),
            # 3rd convolution
            nn.Conv2d(nf2, nf3, ks3),
            # Activation function
            nn.ReLU(inplace=True),
            # 1st reduction
            nn.MaxPool2d(rd1),
            # 1st dropout
            nn.Dropout(dp1)
        )

        # Determine the height, width, and total size of the feature map
        feat_map_height = (self.dim_in - 6) // 2
        feat_map_width = (self.frame_width - 6) // 2
        self.conv_embedding_size = nf3 * feat_map_height * feat_map_width

        # Number of neurons for each fully-connected stage
        self.fc_embedding_size = 128 * self.model_complexity

        # Extract tablature parameters
        num_groups = self.profile.get_num_dofs()
        num_classes = self.profile.num_pitches + 1

        self.dense = nn.Sequential(
            # 1st fully-connected
            nn.Linear(self.conv_embedding_size, self.fc_embedding_size),
            # Activation function
            nn.ReLU(inplace=True),
            # 2nd dropout
            nn.Dropout(dp2),
            # 2nd fully-connected
            SoftmaxGroups(self.fc_embedding_size, num_groups, num_classes)
        )

    def toggle_online(self):
        """
        Toggle the flag for padding input features. This is necessary to differentiate
        between training/validation, where there is ground-truth for each input frame,
        and inference, where there are no labels to compare against and only one output
        is desired for a group of input features filling the expected frame width.
        """

        # Switch the flag
        self.online = not self.online

    def pre_proc(self, batch):
        """
        Perform necessary pre-processing steps for the transcription model.

        Parameters
        ----------
        batch : dict
          Dictionary containing all relevant fields for a group of tracks

        Returns
        ----------
        batch : dict
          Dictionary with all PyTorch Tensors added to the appropriate device
          and all pre-processing steps complete
        """

        batch = super().pre_proc(batch)

        # Create a local copy of the batch so it is only modified within scope
        # TODO
        # batch = deepcopy(batch)

        # Extract the features from the batch as a NumPy array
        feats = tools.tensor_to_array(batch[tools.KEY_FEATS])
        # Window the features to mimic online/real-time operation
        feats = tools.framify_activations(feats, self.frame_width, pad=(not self.online))
        # Convert the features back to PyTorch tensor and add to device
        feats = tools.array_to_tensor(feats, self.device)
        # Switch the sequence-frame and feature axes
        feats = feats.transpose(-2, -3)
        # Switch the sequence-frame and channel axes
        feats = feats.transpose(-3, -4)

        batch[tools.KEY_FEATS] = feats

        return batch

    def forward(self, feats):
        """
        Perform the main processing steps for TabCNN.

        Parameters
        ----------
        feats : Tensor (B x T x C x F x W)
          Input features for a batch of tracks,
          B - batch size
          T - number of frames
          C - number of channels in features
          F - number of features (frequency bins)
          W - frame width of each sample

        Returns
        ----------
        output : dict w/ Tensor (B x T x O)
          Dictionary containing tablature output
          B - batch size,
          T - number of time steps (frames),
          O - number of output neurons (dim_out)
        """

        # Initialize an empty dictionary to hold output
        output = dict()

        # Obtain the batch size before sequence-frame axis is collapsed
        batch_size = feats.size(0)

        # Collapse the sequence-frame axis into the batch axis,
        # so that each windowed group of frames is treated as one
        # independent sample. This is not done during pre-processing
        # in order to maintain consistency with the notion of batch size
        feats = feats.reshape(-1, self.in_channels, self.dim_in, self.frame_width)

        # Obtain the feature embeddings
        embeddings = self.conv(feats)
        # Flatten spatial features into one embedding
        embeddings = embeddings.flatten(1)
        # Size of the embedding
        embedding_size = embeddings.size(-1)
        # Restore proper batch dimension, unsqueezing sequence-frame axis
        embeddings = embeddings.view(batch_size, -1, embedding_size)

        # Obtain the tablature estimate and add it to the output dictionary
        output[tools.KEY_TABLATURE] = self.dense(embeddings)

        return output

    def post_proc(self, batch):
        """
        Calculate loss and finalize model output.

        Parameters
        ----------
        batch : dict
          Dictionary including model output and potentially
          ground-truth for a group of tracks

        Returns
        ----------
        output : dict
          Dictionary containing tablature as well as loss
        """

        # Extract the raw output
        output = batch[tools.KEY_OUTPUT]

        # Obtain pointers to the output layer
        tablature_output_layer = self.dense[-1]

        # Obtain the tablature estimation
        tablature_est = output[tools.KEY_TABLATURE]

        # Check to see if ground-truth tablature is available
        if tools.KEY_TABLATURE in batch.keys():
            # Extract the ground-truth, calculate the loss and add it to the dictionary
            tablature_ref = batch[tools.KEY_TABLATURE]
            tablature_loss = tablature_output_layer.get_loss(tablature_est, tablature_ref)
            output[tools.KEY_LOSS] = {tools.KEY_LOSS_TOTAL : tablature_loss}

        # Finalize tablature estimation
        output[tools.KEY_TABLATURE] = tablature_output_layer.finalize_output(tablature_est)

        return output

In [ ]:
import torch
import librosa
import numpy as np

# Importiamo gli strumenti della libreria su cui si basa SynthTab
from amt_tools.tools.instrument import GuitarProfile
from amt_tools import tools

# Assicurati di aver eseguito la cella con la classe TabCNN prima di lanciare questo codice!

# ==========================================
# 1. SETUP DEL MODELLO E DEI PESI
# ==========================================
device = torch.device("cpu") # Usa "cuda" se hai una GPU Nvidia configurata
profile = GuitarProfile()    # Crea il profilo standard della chitarra (6 corde)

# Inizializziamo il modello. 192 è il numero standard di bin della CQT per questa rete.
model = TabCNN(dim_in=192, profile=profile, in_channels=1, model_complexity=1, device=device)

# Carichiamo SOLO i pesi fine-tunati (contengono già l'esperienza di SynthTab)
percorso_pesi = "../data/GuitarSet/GuitarSet.pt" 

# 1. Carichiamo il file bypassando il blocco di sicurezza di PyTorch 2.6
file_caricato = torch.load(percorso_pesi, map_location=device, weights_only=False)

# 2. Controllo intelligente: l'autore ha salvato l'intero modello o solo i pesi?
if isinstance(file_caricato, torch.nn.Module):
    # Se ha salvato l'intero modello, estraiamo solo il dizionario dei pesi (state_dict)
    model.load_state_dict(file_caricato.state_dict())
    print("Estratto lo state_dict dall'oggetto modello.")
else:
    # Se è già un dizionario di pesi puro
    model.load_state_dict(file_caricato)
    print("Pesi caricati direttamente.")

model.eval() 
print("Modello e pesi caricati correttamente!")

# ==========================================
# 2. MODULO 1: L'UDITO (Estrazione Feature)
# ==========================================
audio_path = "tuo_file_audio_di_prova.wav" # Sostituisci con un audio reale!

# Carichiamo l'audio a 22050 Hz (il sample rate su cui è stata addestrata la rete)
audio, sr = librosa.load(audio_path, sr=22050)

# Calcoliamo la CQT (Constant-Q Transform)
# Questi sono i parametri standard esatti usati in letteratura per TabCNN
cqt = librosa.cqt(audio, 
                  sr=sr, 
                  hop_length=512, 
                  fmin=librosa.note_to_hz('E2'), # La frequenza del Mi basso (circa 82.4 Hz)
                  n_bins=192, 
                  bins_per_octave=24)

# Prendiamo la magnitudo (valori assoluti) dello spettrogramma
cqt_mag = np.abs(cqt)

# amt_tools si aspetta la forma (Canali, Features, Frames) -> (1, 192, Tempo)
cqt_tensor = torch.tensor(cqt_mag, dtype=torch.float32).unsqueeze(0).to(device)
print(f"Audio processato. Shape della CQT: {cqt_tensor.shape}")

# ==========================================
# 3. MODULO 2: IL CERVELLO (Predizione)
# ==========================================
# Creiamo il dizionario "batch" fittizio come se lo aspetta il codice originale
batch = {tools.KEY_FEATS: cqt_tensor}

# Disabilitiamo il calcolo dei gradienti per risparmiare moltissima RAM
with torch.no_grad():
    # 3a. Pre-processing: Il modello taglia la CQT in finestre temporali sovrapposte (framing)
    batch = model.pre_proc(batch)
    
    # 3b. Forward Pass: L'audio passa attraverso le convoluzioni!
    output = model(batch[tools.KEY_FEATS])
    
    # 3c. Post-processing: Il modello applica la funzione Softmax per calcolare le probabilità finali
    batch[tools.KEY_OUTPUT] = output
    risultato_finale = model.post_proc(batch)

# ==========================================
# 4. RISULTATI (Pronti per il DTW)
# ==========================================
tablatura_stimata = risultato_finale[tools.KEY_TABLATURE]

print("\n--- INFERENZA COMPLETATA ---")
print(f"Shape del tensore in uscita: {tablatura_stimata.shape}")
print("Spiegazione dell'output:")
print("- Dimensione 0: Batch (1 audio)")
print("- Dimensione 1: Numero di frame temporali")
print("- Dimensione 2: Matrice Corde x Tasti (Probabilità predette)")

UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL tabcnn.TabCNN was not an allowed global by default. Please use `torch.serialization.add_safe_globals([tabcnn.TabCNN])` or the `torch.serialization.safe_globals([tabcnn.TabCNN])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn more about types accepted by default with weight